# Predicao do peso de animais por regressao

Objetivo: treinar um modelo de regressao para estimar o `PESO` do animal a partir das caracteristicas disponiveis na base.

## 1. Importacao das bibliotecas

In [ ]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

## 2. Carregamento dos dados

In [ ]:
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = ROOT_DIR / 'data' / 'raw' / 'dados_ultrassom_animais.csv'

df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
df.shape

In [ ]:
(df.isna().mean() * 100).sort_values(ascending=False).head(20)

## 3. Pre-processamento

- O rotulo (`y`) sera a coluna `PESO`.
- Colunas sem nome e colunas com muitos dados faltantes serao removidas.
- Numeros com virgula decimal serao convertidos para formato numerico.
- Identificadores serao removidos para evitar que o modelo memorize registros.

In [ ]:
TARGET = 'PESO'
MISSING_THRESHOLD = 0.50
SELECTED_FEATURES = [
    'AC', 'AG', 'CC', 'AP', 'P.C', 'CT', 'CO', 'CCAB',
    'LIL', 'LIS', 'Cga', 'Cper', 'PerPe', 'Ccau', 'DC',
]

def convert_decimal_columns(dataframe):
    dataframe = dataframe.copy()
    for column in dataframe.columns:
        if pd.api.types.is_object_dtype(dataframe[column]) or pd.api.types.is_string_dtype(dataframe[column]):
            values = dataframe[column].astype('string').str.strip()
            numeric_values = pd.to_numeric(
                values.str.replace('.', '', regex=False).str.replace(',', '.', regex=False),
                errors='coerce',
            )
            original_not_null = values.notna().sum()
            converted_not_null = numeric_values.notna().sum()
            if original_not_null > 0 and converted_not_null / original_not_null >= 0.80:
                dataframe[column] = numeric_values
    return dataframe

def clean_data(dataframe):
    dataframe = dataframe.copy()
    dataframe = dataframe.drop(columns=[col for col in dataframe.columns if col.startswith('Unnamed')], errors='ignore')
    dataframe = convert_decimal_columns(dataframe)
    dataframe = dataframe.dropna(subset=[TARGET])
    missing_ratio = dataframe.isna().mean()
    dataframe = dataframe.drop(columns=missing_ratio[missing_ratio > MISSING_THRESHOLD].index, errors='ignore')
    dataframe = dataframe[[*SELECTED_FEATURES, TARGET]].copy()
    return dataframe

clean_df = clean_data(df)
clean_df.head()

In [ ]:
clean_df.shape

## 4. Divisao treino/teste

A divisao sera de 70% para treino e 30% para teste.

In [ ]:
X = clean_df.drop(columns=[TARGET])
y = clean_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
)

X_train.shape, X_test.shape

## 5. Pipeline de modelagem

In [ ]:
numeric_features = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=['number']).columns.tolist()

numeric_pipeline = Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))])
categorical_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore')),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_pipeline, numeric_features),
        ('categorical', categorical_pipeline, categorical_features),
    ]
)

In [ ]:
models = {
    'Regressao Linear': LinearRegression(),
}

results = []
trained_models = {}

for name, estimator in models.items():
    model = Pipeline(steps=[('preprocessor', preprocessor), ('model', estimator)])
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    trained_models[name] = model
    results.append({
        'modelo': name,
        'mae': mean_absolute_error(y_test, predictions),
        'rmse': root_mean_squared_error(y_test, predictions),
        'r2': r2_score(y_test, predictions),
    })

metrics = pd.DataFrame(results).sort_values('rmse')
metrics

## 6. Salvamento dos resultados

In [ ]:
best_model_name = metrics.iloc[0]['modelo']
best_model = trained_models[best_model_name]

(ROOT_DIR / 'data' / 'processed').mkdir(parents=True, exist_ok=True)
(ROOT_DIR / 'models').mkdir(parents=True, exist_ok=True)
(ROOT_DIR / 'reports').mkdir(parents=True, exist_ok=True)

clean_df.to_csv(ROOT_DIR / 'data' / 'processed' / 'dados_limpos.csv', index=False)
metrics.to_csv(ROOT_DIR / 'reports' / 'metricas_modelo.csv', index=False)
joblib.dump(best_model, ROOT_DIR / 'models' / 'modelo_peso.joblib')

best_model_name